<hr style="border: 6px solid#003262;" />

<div align="center">
    <img src="assets/content/images/thumbnail.png" align="center" width="20%">
</div>

<br>

# SYNTHETIC DATA FOR CHURN AND CONTRIBUTION MODELING

<br>

**About:** This notebook shows how to bootstrap a realistic tabular dataset from a small seed CSV using pandas and NumPy, then use the generated data to model two decisions that non-profit fundraising teams care about: which donors are at risk of churning, and how large the next contribution from a donor is likely to be.

**Learning Goals:** By the end of this notebook you can (1) generate synthetic tabular records from a seed distribution using custom pandas/NumPy sampling functions, (2) train and compare several classifiers (Gradient Boosting, Logistic Regression, Decision Trees, Random Forest) for churn prediction, and (3) fit regression models (Linear and Logistic Regression) that estimate donor contributions from profile features.

**Keywords:** synthetic data, churn analysis, classification, regression, pandas

**Prerequisite Knowledge:** (1) Python, (2) Pandas, (3) Matplotlib, (4) Basic sklearn

**Target User:** Applied data scientists and analysts working on donor, customer, or membership data where the real dataset is small, sensitive, or slow to obtain.


<hr style="border: 4px solid#003262;" />

<a name='Part_table_contents' id="Part_table_contents"></a>


#### CONTENTS

> #### [PART 1: SYNTHETIC DATA GENERATION](#Part_1)
> #### [PART 2: CHURN ANALYSIS](#Part_2)
> #### [PART 3: PREDICTING CONTRIBUTIONS](#Part_3)

<br>


In [ ]:
# import libraries and packages
import pandas as pd
from pandas import DataFrame
import numpy as np
import datetime as dt
from datetime import datetime,tzinfo
from pytz import timezone
import time
import pytz

# import  ML libraries and packages
from sklearn.preprocessing import LabelEncoder

# for random generator
import random

# plotting and pretty print
import matplotlib.pyplot as plt

# pip install plotly
# pip install cufflinks
import plotly.offline as py
import plotly.graph_objs as go
import plotly.figure_factory as ff
py.init_notebook_mode(connected=True)
import csv
import seaborn as sns
sns.set(style="whitegrid")

# do not display warnings
import warnings
warnings.filterwarnings('ignore')

<a id='Part_1'></a>

<hr style="border: 2px solid#003262;" />

#### PART 1

## **SYNTHETIC DATA** GENERATION FROM A SEED **CSV**

The starting point is a small seed CSV of real intern-dropout style records. Rather than using it directly, we treat each column as an empirical distribution and sample from those distributions to bootstrap a larger synthetic table. Two guarantees matter: sampled columns should preserve the marginal look of the seed (roughly the same range and shape), and the process should be reproducible.

We build a small library of type-aware generators - one for booleans, one for integers, one for floats, one for categorical strings, and one for label-encoded categoricals - then compose them into a 500-row synthetic dataset that carries a plausible churn signal.

___

**Note:** The seed file `data_seeds/intern_dropouts_seed.csv` must be present in the working directory. Column names and types drive the generator choices below.

___

Sources consulted:
- pandas `DataFrame.sample` docs, https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.sample.html
- NumPy random sampling reference, https://numpy.org/doc/stable/reference/random/index.html


In [ ]:
df = pd.read_csv('data_seeds/intern_dropouts_seed.csv')
df

<a id='Part_1_1'></a>

<hr style="border: 1px solid#003262;" />

#### PART 1.1: TYPE-AWARE GENERATOR FUNCTIONS

Each function below returns a synthetic column of length `n` conditioned on the seed column's observed values. Keeping them small and single-purpose makes it easy to swap in a different distributional assumption later (e.g., swap uniform floats for a fitted normal) without rewriting the assembly step.


In [ ]:
def gen_bin_data(df, low, split, n):
    '''
        Generates list of random 0's and 1's
        @param df is list of binary data
        @param low the lower end of the bin distribution (int)
        @param split the upper end of the bin distribution (int)
        @param n is loop stop (int)
        @return is a list containing 0's and 1's of length n
    '''
    # redundancy here to ensure count is same
    n = n
    for i in range(n):        
        sampl_array = np.random.randint(low,split, size=20)
        sampl = sampl_array.tolist()
        df += sampl
    return(df)

print(gen_bin_data([], 10, 20, 1))

In [ ]:
def gen_int_data(df, low, high, n):
    '''
        Generates list of random integers from low (inclusive) to high (exclusive) and appends iterably
        @param df is list of int data
        @param low is start (int)
        @param high is stop (int)
        @param n is loop stop (int)
        @return is a list of length n
    '''
    # ensure count is same
    count = ((n)*4)
    for i in range(count):        
        sampl = random.sample(range(low, high), k=5)
        df += sampl
    
    for n, i in enumerate(df):
        if i == low:
            df[n] = random.randint(low+1,high-1)       
    return(df)

print(gen_int_data([], 1, 20, 1))

In [ ]:
def gen_flt_data(df, low, high, n):
    '''
        Generates nparray of random floats from low (inclusive) to high (inclusive) and appends iterably
        @param df is list of floats
        @param low is start (int)
        @param hight is stop (int)
        @param n is loop stop (int)
        @return is a list of length n containing random floats
    '''
    # ensure count is same
    count = ((n+1)*20)-10
    df = []
    for i in range(count): 
        # pick a float in [low, high] and then round it to one decimal
        sampl_array = round(random.uniform(low, high),1)
        df = np.append(df, sampl_array)
    
    np_df = np.asarray(df)
    return(np_df)

print(gen_flt_data(x2, 1.5, 4.1, rows//10))

In [ ]:
def gen_str_data(df, n):
    '''
        Generates list containing random sampling of existing string elements and appends iterably
        @param df is list containing str data
        @param n is loop stop (int)
        @return is a list containing sampling with replacement of n consecutive samples each of length 20.
    '''

    # redundancy here to ensure count is same    
    n = n
    # extract unique entries
    df_set = set(df)
    df_list = list(df_set)
    for i in range(n):        
        # pick 20 elements at random from the list 
        sampl = random.choices(df_list, k = 20)
        df += sampl
    return(df)

In [ ]:
def label_enc(df):
    '''
        Encodes integer vector from list of string elements (using scikit-learn)
        @param df is list of str elements
        @return is an array of one-hot encodings.
    '''

    from sklearn import preprocessing
    le = preprocessing.LabelEncoder()
    # integer encode
    integer_encoded = le.fit_transform(np.asarray(df))
    return(integer_encoded)

In [ ]:
def change_split(df, att, split=1):
    '''
        Modifies list containing random sampling of existing string elements according to ratio split
        @param df is a list of str elements
        @param att is the string you want to increase
        @param split is the ratio of att:1 you want to influence
        @return is a list of strings with split:1 ratio -- as opposed to default 1:1
    '''
    
    # split denotes the n:1 ratio to be considered
   
    from itertools import chain
    import types
        
    choice = list(set(df))    # list of unique elements ['x', 'y']
    percent = len(df)//(int(len(choice)))
    
    # extract first percent items from the list
    df_percent = df[:percent]
    for n, i in enumerate(df_percent):
        if i != att: 
            df_percent[n] = random.choices(choice, k = 1)[0]

    # put list back together
    df = df[percent:] + df_percent
    
    return(df)

<!--Navigate back to table of contents-->
<div alig="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents">
            Table of Contents
        </a>
    </span>
</div>
<!-------------------------------------->

<a id='Part_1_2'></a>

<hr style="border: 1px solid#003262;" />

#### PART 1.2: ASSEMBLING A 500-ROW SYNTHETIC TABLE

We draw 500 records - fifty times the seed size - by calling each generator with the corresponding seed column. The final `pd.DataFrame` combines the columns into a single table that downstream sections treat as the working dataset.


In [ ]:
rows = 500
n = int(rows/10) 
    
# extract rows as lists for data generation
x1 = list(df['Distance'])
x2 = list(df['GPA'])
x3 = list(df['Class'])
x4 = list(df['Gender'])
x5 = list(df['Ethnicity'])
x6 = list(df['Low-income'])
x7 = list(df['Sector'])
y  = list(df['Churned']) 

# create >1000 rows of content
f1 = gen_int_data(x1, 0, 21, n)
f2 = gen_flt_data(x2, 1.5, 4.1, n)
f_3 = gen_str_data(x3, n)
f_4 = gen_str_data(x4, n)
f_5 = gen_str_data(x5, n)
f_6 = gen_str_data(x6, n)
f_7 = gen_str_data(x7, n)
ry = gen_bin_data(y, 0, 2, n)

# change distribution based on attribute and split
f_4 = change_split(f_4, "Female", 2)
f_5 = change_split(f_5, "Caucasian", 4)
f_6 = change_split(f_6, "No", 5)

# convert strings to label-encoded objects
f3 = label_enc(f_3)
f4 = label_enc(f_4)
f5 = label_enc(f_5)
f6 = label_enc(f_6)
f7 = label_enc(f_7)


In [ ]:
# sanity check
print(len(ry), len(f1), len(f2), len(f3), len(f4), len(f5), len(f6), len(f7))

In [ ]:
# combine lists into pandas dataframe
data = {'DISTANCE':f1, 'GPA':f2, 'CLASS':f3, 'GENDER':f4,'ETHNICITY':f5, 'LOW-INCOME':f6, 'SECTOR':f7, 'CHURN':ry}
cdf = pd.DataFrame(data)

# sumary statistics for all features except CHURN (binary)
cdf.loc[:, cdf.columns != 'CHURN'].describe()

<a id='cc_1'></a>

<hr style="border: 2px solid#003262;" />

<!--Concept Check banner-->
<div align="left" style="text-align: left; background-color:#003262;">
    <span>
        <hr style="border: 8px solid#003262;" />
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:0px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:24px;letter-spacing:0px;line-height:20px;padding:24px 40px;text-align:left;text-decoration:none; align:left">
            <strong>CONCEPT</strong> CHECK 1
        </a>
    </span>
</div>
<!-------------------------------------->

<br>

**Question:** The five generators above each preserve a different aspect of the seed distribution. Pick one continuous column of the seed (e.g., `GPA`) and write a short function `gen_normal_data(df, col, n)` that samples `n` values from a Gaussian fit to the seed column, then plot a histogram overlay of the seed values and your synthetic values. Comment on whether the Gaussian assumption is a good fit for that column.


In [ ]:
### YOUR CODE HERE ###
import numpy as np
import matplotlib.pyplot as plt

def gen_normal_data(df, col, n):
    mu = ...
    sigma = ...
    return ...

# synthetic = gen_normal_data(df, 'GPA', 500)
# plt.hist(df['GPA'], bins=15, alpha=0.5, label='seed')
# plt.hist(synthetic, bins=15, alpha=0.5, label='synthetic')
# plt.legend(); plt.show()


<!--Navigate back to table of contents-->
<div alig="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents">
            Table of Contents
        </a>
    </span>
</div>
<!-------------------------------------->

<a id='Part_2'></a>

<hr style="border: 2px solid#003262;" />

#### PART 2

## **CHURN** ANALYSIS ON THE SYNTHETIC **DONOR** DATASET

Churn is framed here as a binary outcome per donor: `CHURN = 0` denotes a donor who stopped contributing, `CHURN = 1` denotes a retained donor. We inspect the marginal shape of each feature, look at the correlation structure, establish a baseline accuracy from the majority class, then train four classifiers and compare their held-out accuracy.

The four classifiers were chosen to span the bias-variance spectrum: Logistic Regression as a linear baseline, Decision Trees as a single interpretable non-linear model, Random Forest as a variance-reducing ensemble of trees, and Gradient Boosting as a bias-reducing sequential ensemble.

___

Sources consulted:
- scikit-learn user guide, ensemble methods, https://scikit-learn.org/stable/modules/ensemble.html
- scikit-learn logistic regression reference, https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression

<!-- TODO: verify against current sklearn docs -->


<a id='Part_2_1'></a>

<hr style="border: 1px solid#003262;" />

#### PART 2.1: EXPLORATORY DATA ANALYSIS

Before fitting any model, we look at per-feature histograms and the pairwise correlation matrix. This surfaces the two facts every downstream model depends on: whether any feature is nearly constant (and therefore useless), and whether any two features carry the same information (and therefore invite multicollinearity).


In [ ]:
#### Histograms ####
plot_data = {'CHURN':ry, 'DISTANCE':f1, 'GPA':f2, 'CLASS':f_3, 'GENDER':f_4,'ETHNICITY':f_5, 'LOW-INCOME':f_6, 'SECTOR':f_7}
plotdf = pd.DataFrame(plot_data)

fig = plt.subplots(figsize=(15,20))
for i, j in enumerate(list(plotdf.columns)):
    plt.subplot(4,2,i+1)
    plt.subplots_adjust(hspace = 1.0)
    sns.countplot(x=j, data = plotdf, hue = 'CHURN')
    plt.xticks(rotation=90)
    plt.xlabel(xlabel=plotdf.columns[i])
    plt.title("Histogram of TIPS Churn Features")

# save image to pwd as png
plt.savefig('images/eda.png')


In [ ]:
#### Correlation Matrix ####

corr = cdf.corr()
sns.heatmap(corr, 
            xticklabels=corr.columns.values, 
            yticklabels=corr.columns.values, 
            annot = True, 
            annot_kws={'size':12},
            cmap= sns.diverging_palette(240, 21, as_cmap=True))
heat_map=plt.gcf()
heat_map.set_size_inches(15,10)
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)
# plt.show()

# save image to pwd as png
plt.savefig('images/corrogram.png')

<br>

**Baseline Accuracy**

Before training any model we compute the trivial-classifier accuracy: predict the majority class for every row. Any model that fails to beat this number is not learning anything useful from the features.


In [ ]:
ch=0
for i in cdf.CHURN:
    if i != 0:
        ch += 1
        
print("Baseline Accuracy: " , (ch/1010) ,"\n" )

<br>

**Categorical Structure via KMeans**

To sanity check whether the categorical features carry any group structure at all, we run KMeans on the numeric encoding. This is not a model of churn - it is a diagnostic that tells us whether the feature space clusters at all.


In [ ]:
# Explore the influce of 'categories' on those students that left early
from sklearn.cluster import KMeans

# create clusters
churn_student = cdf[['DISTANCE', 'GPA']][cdf.CHURN == 1]
kmeans = KMeans(3, random_state = 0).fit(churn_student)

# display findings

churn_student['label'] = kmeans.labels_
plt.scatter(churn_student['DISTANCE'], 
            churn_student['GPA'],
            c=churn_student['label'], 
            cmap='Accent')
plt.xlabel('Distance (mi)')
plt.ylabel('Grade Point Average')
plt.title("Three Clusters of Students that Churned")
plt.show()

<!--Navigate back to table of contents-->
<div alig="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents">
            Table of Contents
        </a>
    </span>
</div>
<!-------------------------------------->

<a id='Part_2_2'></a>

<hr style="border: 1px solid#003262;" />

#### PART 2.2: TRAIN AND COMPARE FOUR CLASSIFIERS

We split the data, then fit four classifiers on the same train/test partition so accuracies are directly comparable. Each block below trains one model, prints held-out accuracy, and (for tree-based models) inspects feature importance or the fitted tree.


**Split data for classification**

In [ ]:
#### prepare data for split -- features and response ####
from sklearn import preprocessing

X=cdf[['DISTANCE', 'GPA', 'CLASS', 'GENDER', 'ETHNICITY', 'LOW-INCOME', 'SECTOR']]
y=cdf['CHURN']

# Import train_test_split function
from sklearn.model_selection import train_test_split

# Split dataset into training set and test set
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=42) 


<br>

**Model 1: Gradient Boosting Classifier**

In [ ]:
#### Build Model: Gradient Boosting Classifier ####

# Import Gradient Boosting Classifier model
from sklearn.ensemble import GradientBoostingClassifier

# Create Gradient Boosting Classifier
gb = GradientBoostingClassifier()

# Train the model using the training sets
gb.fit(X_train, y_train)

# Predict the response for test dataset
y_pred = gb.predict(X_test)


In [ ]:
#### Evaluate model ####

# module for accuracy calculation
from sklearn import metrics

# Model Accuracy
print("Accuracy:",metrics.accuracy_score(y_test, y_pred))

# Model Precision
print("Precision:",metrics.precision_score(y_test, y_pred))

# Model Recall
print("Recall:",metrics.recall_score(y_test, y_pred))

<br>

**Model 2: Logistic Regression**

In [ ]:
#### Build Model: Logistic Regression ####
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, classification_report

# Use logistic regression to get a model for our data
logisticRegr = LogisticRegression()
logisticRegr.fit(X=X_train, y=y_train)

# Calculate the model's prediction and display it's coeffecients.
test_y_pred = logisticRegr.predict(X_test)
confusion_matrix = confusion_matrix(y_test, test_y_pred)
print('Intercept: ' + str(logisticRegr.intercept_))
print('Regression: ' + str(logisticRegr.coef_))

#### Evaluate Model ####
print('\nAccuracy of logistic regression classifier on test set: {:.2f}'.format(logisticRegr.score(X_test, y_test)))
print("\n", classification_report(y_test, test_y_pred),"\n\n")


In [ ]:
#### Plot Confussion Matrix ####
def plot_confusion_matrix(cm, classes, normalize=False, title='Confusion matrix', cmap=plt.cm.Blues):
    
    import itertools
    
    """
    This function prints and plots the confusion matrix.
    Normalization can be applied by setting `normalize=True`.
    """
    if normalize:
        cm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
        print("\nNormalized confusion matrix")
    else:
        print('\n\nConfusion matrix, without normalization')

    print(cm)

    plt.imshow(cm, interpolation='nearest', cmap=cmap)
    plt.title(title)
    plt.colorbar()
    tick_marks = np.arange(len(classes))
    plt.xticks(tick_marks, classes, rotation=45)
    plt.yticks(tick_marks, classes)

    fmt = '.2f' if normalize else 'd'
    thresh = cm.max() / 2.
    for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
        plt.text(j, i, format(cm[i, j], fmt),
                 horizontalalignment="center",
                 color="white" if cm[i, j] > thresh else "black")

    plt.ylabel('True label')
    plt.xlabel('Predicted label\n\n')
    plt.tight_layout()


# Compute confusion matrix
np.set_printoptions(precision=2)

# Plot non-normalized confusion matrix
class_names = ['No Churn', "Churn"]
plt.figure()
plot_confusion_matrix(confusion_matrix, classes= class_names,
                      title='\n\nConfusion matrix, without normalization')

# Plot normalized confusion matrix
plt.figure()
plot_confusion_matrix(confusion_matrix, classes=class_names, normalize=True,
                      title='Normalized confusion matrix')

plt.show()

<br>

**Feature importance and tree visualization helpers**

In [ ]:
#### Plot of Feature Importance ####

"""
    Plots histograms of features sorted by highest influence
    @param model is fitted model to be evalued
    @param model_name is string name used for plot title
"""

def feature_importance(model, model_name):
    # calculate feature importance
    importance = model.feature_importances_
    # decrementally sort feature importance
    indices = np.argsort(importance)[::-1]
    # sort features names to match sorted importance
    names = [X.columns[i] for i in indices]
    
    plt.figure()
    plt.title("Feature Importance " + model_name)
    plt.bar(range(X.shape[1]), importance[indices], color="lightblue")
    # bar names
    plt.xticks(range(X.shape[1]), names, rotation = 90)
    
    plt.show(names,importance)

In [ ]:
#### Plot Trees ####

"""
    Generates a PDF visual of tree
    #!conda install python-graphviz
    @param t_model is fitted model e.g. decisionTree
"""

def plot_tree(t_model):
    plotTree = tree.export_graphviz(decisionTree, out_file=None, 
                         feature_names = list(X_train.columns.values),  
                         class_names = ['No churn', 'Churn'],
                         filled=True, rounded=True,  
                         special_characters=True)  
    graph = graphviz.Source(plotTree)
    graph.render('decision_tree', view=True)

<br>

**Model 3: Decision Tree**

In [ ]:
#### Build Model: Decision Trees ####

from sklearn import tree
import graphviz 
 
# Create each decision tree (pruned and unpruned)
decisionTree_unpruned = tree.DecisionTreeClassifier()
decisionTree = tree.DecisionTreeClassifier(max_depth = 4)
 
# Fit each tree to our training data
decisionTree_unpruned = decisionTree_unpruned.fit(X=X_train, y=y_train)
decisionTree = decisionTree.fit(X=X_train, y=y_train)

#### plot tree ####
plot_tree(decisionTree)

In [ ]:
#### Trim Decision Tree ####
test_y_pred_dt = decisionTree.predict(X_test)

#### Evaluate Model ####
print('Accuracy of decision tree classifier on test set: {:.2f}'.format(decisionTree.score(X_test, y_test)))

#### Plot Feature Importance ####
feature_importance(decisionTree, "for Decision Trees Classifier")

<br>

**Model 4: Random Forest**

In [ ]:
#### Build Model: Random Forrest ####
from sklearn.ensemble import RandomForestClassifier
randomForest = RandomForestClassifier()
randomForest = randomForest.fit(X_train, y_train)

#### Evaluate Model ####
print('Accuracy of random forest classifier on test set: {:.2f}'.format(randomForest.score(X_test, y_test)))

#### Plot Tree ####
plot_tree(randomForest)

#### Plot Feature Importance ####
feature_importance(randomForest, "for Random Forrest Classifier")

In [ ]:
from sklearn import tree
i_tree = 0
for tree_in_forest in randomForest.estimators_:
    with open('tree_' + str(i_tree) + '.dot', 'w') as my_file:
        my_file = tree.export_graphviz(tree_in_forest, out_file = my_file)#TODO?
    i_tree = i_tree + 1

<a id='cc_2'></a>

<hr style="border: 2px solid#003262;" />

<!--Concept Check banner-->
<div align="left" style="text-align: left; background-color:#003262;">
    <span>
        <hr style="border: 8px solid#003262;" />
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:0px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:24px;letter-spacing:0px;line-height:20px;padding:24px 40px;text-align:left;text-decoration:none; align:left">
            <strong>CONCEPT</strong> CHECK 2
        </a>
    </span>
</div>
<!-------------------------------------->

<br>

**Question:** All four classifiers were compared on a single train/test split, which means every reported accuracy is noisy. Rewrite the comparison to use `sklearn.model_selection.cross_val_score` with 5-fold cross validation, then produce a small DataFrame that lists each model's mean and standard deviation of accuracy across folds. Which model wins once you account for fold-to-fold variance?


In [ ]:
### YOUR CODE HERE ###
from sklearn.model_selection import cross_val_score
import pandas as pd

models = {
    'GradientBoosting': ...,
    'LogisticRegression': ...,
    'DecisionTree': ...,
    'RandomForest': ...,
}

# results = []
# for name, model in models.items():
#     scores = cross_val_score(model, X, y, cv=5, scoring='accuracy')
#     results.append({'model': name, 'mean_acc': scores.mean(), 'std_acc': scores.std()})
# pd.DataFrame(results).sort_values('mean_acc', ascending=False)


<!--Navigate back to table of contents-->
<div alig="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents">
            Table of Contents
        </a>
    </span>
</div>
<!-------------------------------------->

<a id='Part_3'></a>

<hr style="border: 2px solid#003262;" />

#### PART 3

## **PREDICTING** DONOR **CONTRIBUTIONS**

Where Part 2 asked "will this donor stay?", Part 3 asks "how much will this donor give?" That reframes the problem from classification to regression. We reuse the synthetic-data recipe from Part 1 - now applied to a donation-history seed - then fit Linear Regression as a continuous predictor and Logistic Regression as a bucketed comparison.

Sources consulted:
- scikit-learn linear models, https://scikit-learn.org/stable/modules/linear_model.html


<a id='Part_3_1'></a>

<hr style="border: 1px solid#003262;" />

#### PART 3.1: GENERATE THE DONATION DATASET

We load a 20-row donation-history seed and bootstrap it to 500 rows using the same generator functions defined in Part 1.


In [ ]:
df = pd.read_csv('./data_seeds/donation_history_seed.csv')
df

In [ ]:
rows = 500
n = int(rows/20)

# extract rows as lists for data generation
x1 = list(df['Quarter'])
x2 = list(df['Year'])
x3 = list(df['Method'])
x4 = list(df['Luncheon'])
x5 = list(df['Parent'])
x6 = list(df['Alumni'])
x7 = list(df['Board'])
y  = list(df['Amount']) 

# create 500 rows of content
f_1 = gen_str_data(x1, n)
f_2 = gen_str_data(x2, n)
f_3 = gen_str_data(x3, n)
f_4 = gen_str_data(x4, n)
f_5 = gen_str_data(x5, n)
f_6 = gen_str_data(x6, n)
f_7 = gen_str_data(x7, n)
ry = gen_int_data(y, 10, 300, n)

# change distribution based on attribute and split
g1 = change_split(f_1, "Second", 2)
g2 = change_split(f_2, "2018", 5)
g3 = change_split(f_3, "Card", 2)
g4 = change_split(f_4, "Yes", 8)
g5 = change_split(f_5, "Yes", 3)
g6 = change_split(f_6, "Yes", 3)
g7 = change_split(f_7, "Yes", 3)

# convert strings to label-encoded objects
f1 = label_enc(g1)
f2 = label_enc(g2)
f3 = label_enc(g3)
f4 = label_enc(g4)
f5 = label_enc(g5)
f6 = label_enc(g6)
f7 = label_enc(g7)

In [ ]:
print(len(f1))
print(len(f2))
print(len(f3))
print(len(f4))
print(len(f5))
print(len(f6))
print(len(f7))
print(len(ry))

In [ ]:
# combine lists into pandas dataframes
data = {'AMOUNT':ry, 'QUARTER':f1, 'YEAR':f2, 'METHOD':f3, 'LUNCHEON':f4,'PARENT':f5, 'ALUMNI':f6, 'BOARD':f7}
rdf = pd.DataFrame(data)

rdf.describe()

<!--Navigate back to table of contents-->
<div alig="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents">
            Table of Contents
        </a>
    </span>
</div>
<!-------------------------------------->

<a id='Part_3_2'></a>

<hr style="border: 1px solid#003262;" />

#### PART 3.2: EXPLORE THE DONATION FEATURES

We repeat the EDA pattern from Part 2 (histograms, box/violin plots, correlation matrix) but now the target `AMOUNT` is continuous, so we look at its distribution shape and how each feature relates to it.


In [ ]:
bucket_ry = pd.cut(rdf.AMOUNT, bins=5)
plot_data = {'AMOUNT':ry, 'QUARTER':g1, 'YEAR':g2, 'METHOD':g3, 'LUNCHEON':g4}
plotdf = pd.DataFrame(plot_data)

In [ ]:
#### Boxplots ####

# Fixing random state for reproducibility
np.random.seed(115)

# Horizontal violin plot
fig, axes = plt.subplots(figsize=(15,20))
for i, j in enumerate(list(plotdf.columns)):
    plt.subplot(4,2,i+1)
    plt.subplots_adjust(hspace = 1.0)
    sns.violinplot(x=j, y = "AMOUNT", data = plotdf, hue="LUNCHEON")
    plt.xticks(rotation=90)
    plt.title("Violin Plot of TIPS Regression Features")

# save image to pwd as png
plt.savefig('images/regeda.png')

In [ ]:
#### Histograms ####

fig = plt.subplots(figsize=(15,20))
for i, j in enumerate(list(plotdf.columns)):
    plt.subplot(4,2,i+1)
    plt.subplots_adjust(hspace = 1.0)
    sns.countplot(x=j, data = plotdf, hue = 'LUNCHEON')
    plt.xticks(rotation=90)
    plt.title("Histogram of TIPS Regression Features")

# save image to pwd as png
plt.savefig('images/regeda2.png')

In [ ]:
#### Correlation Matrix ####

corr = rdf.corr()
sns.heatmap(corr, 
            xticklabels=corr.columns.values, 
            yticklabels=corr.columns.values, 
            annot = True, 
            annot_kws={'size':12},
            cmap= sns.diverging_palette(220, 15, as_cmap=True)) # TODO?
heat_map=plt.gcf()
heat_map.set_size_inches(15,10)
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)
# plt.show()

# save image to pwd as png
plt.savefig('images/corrogram2.png')

<!--Navigate back to table of contents-->
<div alig="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents">
            Table of Contents
        </a>
    </span>
</div>
<!-------------------------------------->

<a id='Part_3_3'></a>

<hr style="border: 1px solid#003262;" />

#### PART 3.3: BUILD CONTRIBUTION MODELS

We engineer a compact `PERIOD` feature that zips year and quarter into a single ordinal, split the time-sorted data, and fit two models: Linear Regression on the raw amount, and Logistic Regression on a bucketed amount. Comparing them shows the classic tradeoff - Linear Regression preserves granularity, Logistic Regression trades granularity for calibrated class probabilities.


**Feature engineering: `PERIOD`**

In [ ]:
# Zip year and quarter into single feature
period =[str(m)+str(n) for m,n in zip(g2,f1)]

per = label_enc(period)

pdata = {'AMOUNT':ry, 'PERIOD':per, 'METHOD':f3, 'LUNCHEON':f4,'PARENT':f5, 'ALUMNI':f6, 'BOARD':f7}
pdf = pd.DataFrame(pdata)
pdf.head()

**Split time-stamped data**

In [ ]:
from sklearn import preprocessing

#### Sort Time Stamped Data ####
pdf.sort_values(by=['PERIOD'], inplace=True)

#### prepare data for split -- features and response ####
X=pdf[['PERIOD', 'METHOD', 'LUNCHEON', 'PARENT', 'ALUMNI', 'BOARD']]
pdf['LOG_AMOUNT'] = np.log10(pdf['AMOUNT'])
y=pdf['LOG_AMOUNT']

# Import train_test_split function
from sklearn.model_selection import train_test_split

# Split dataset into training set and test set
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=115) 


In [ ]:
sns.pairplot(pdf, hue = "METHOD")
# save image to pwd as png
plt.savefig('images/Regr_Corr_Plot.png')

In [ ]:
# Sanity check -- ensure dimensions match
print(len(X_test), len(y_test))
print(len(y_pred))

<br>

**Model: Linear Regression on contribution amount**

In [ ]:
#### Build Model: Linear Regression ####

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error,r2_score

# Train the model
LinearRegr = LinearRegression()
LinearRegr.fit(X_train, y_train)

# Make predictions using the testing set
y_pred = LinearRegr.predict(X_test)

#### Explore Model ####
# The intercept
print('\nIntercept: ', LinearRegr.intercept_)

# The coefficients
print('\nCoefficients:', LinearRegr.coef_)

# Explained variance score: 1 is perfect prediction
print('\nVariance score: %.2f' % r2_score(y_test, y_pred))


#### Evaluate Model ####
# MAE
print("\n\nMAE: %.23f" % mean_absolute_error(y_test, y_pred))

# MSE
print("\nMSE: %.23f" % mean_squared_error(y_test, y_pred))

# MAE
print("\nRMSE: %.23f" % np.sqrt(mean_absolute_error(y_test, y_pred)))

<br>

**Model: Logistic Regression on bucketed contribution**

In [ ]:
from sklearn.linear_model import LogisticRegression

# Train the model
LogRegr = LogisticRegression(random_state=0, solver='lbfgs', multi_class='multinomial')
LogRegr.fit(X_train, y_train)

# Make predictions using the testing set
y_pred = LogRegr.predict_proba(X_test)

#### Explore Model ####
# The intercept
print('\nIntercept: ', LogRegr.intercept_)

# The coefficients
print('\nCoefficients:', LogRegr.coef_)

# Explained variance score: 1 is perfect prediction
print('\nVariance score: %.2f' % r2_score(y_test, y_pred))


#### Evaluate Model ####
# MAE
print("\n\nMAE: %.23f" % mean_absolute_error(y_test, y_pred))

# MSE
print("\nMSE: %.23f" % mean_squared_error(y_test, y_pred))

# MAE
print("\nRMSE: %.23f" % np.sqrt(mean_absolute_error(y_test, y_pred)))

<a id='cc_3'></a>

<hr style="border: 2px solid#003262;" />

<!--Concept Check banner-->
<div align="left" style="text-align: left; background-color:#003262;">
    <span>
        <hr style="border: 8px solid#003262;" />
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:0px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:24px;letter-spacing:0px;line-height:20px;padding:24px 40px;text-align:left;text-decoration:none; align:left">
            <strong>CONCEPT</strong> CHECK 3
        </a>
    </span>
</div>
<!-------------------------------------->

<br>

**Question:** Linear Regression on `AMOUNT` reports an R-squared (or MSE) on the held-out data, but this hides where the model is wrong. Compute the residuals `y_test - y_pred` for the Linear Regression fit above, then produce a residual plot (residual on the y-axis, predicted amount on the x-axis). Describe one pattern you see in the residuals and explain what it implies about a missing feature or a violated assumption.


In [ ]:
### YOUR CODE HERE ###
import matplotlib.pyplot as plt

residuals = ...   # y_test - y_pred

# plt.figure(figsize=(8, 5))
# plt.scatter(y_pred, residuals, alpha=0.6)
# plt.axhline(0, color='red', linestyle='--')
# plt.xlabel('Predicted amount')
# plt.ylabel('Residual (actual - predicted)')
# plt.title('Residuals vs. predicted contribution')
# plt.show()

# Written interpretation:
# ...


<!--Navigate back to table of contents-->
<div alig="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents">
            Table of Contents
        </a>
    </span>
</div>
<!-------------------------------------->

<hr style="border: 2px solid#003262;" />

## WHAT COMES NEXT

The recipe used here - hand-write per-column generators, then compose them into a synthetic table - works well when the seed data has a clear schema and you want the result to look like a domain-specific dataset. But it does not scale: writing generators is manual work, and the result is only as expressive as the assumptions baked into each function.

The companion notebook, `02_synthetic_data_generation_sklearn.ipynb`, takes the opposite approach: it uses scikit-learn's `make_regression`, `make_classification`, `make_blobs`, and `make_circles` to generate benchmark datasets with tunable difficulty. Those generators are the right tool when you need a synthetic dataset to stress-test a model rather than to represent a specific real-world domain.

<hr style="border: 6px solid#003262;" />
